# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima8211/ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis + time window

One row represents one content page for a defined observation window. For this assignment, I will use page-level search performance over a 90-day window, with recent and previous 30-day periods used when comparing performance changes. The goal is to identify pages whose search performance is declining and prioritize them for review or refresh
.

In [3]:
# This cell is for CODE (numbers, a query, a check).
import os
import sys
import subprocess
import pandas as pd

REPO_URL = "https://github.com/Fatima8211/ML_Internship.git"
REPO_DIR = "/content/ML_Internship"

# Clone your repository if it is not already available
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

# Load starter dataset
csv_path = "data/raw/content_refresh_anonymized.csv"

print("Dataset exists:", os.path.exists(csv_path))

df = pd.read_csv(csv_path)

print("Rows:", len(df))
print("Unique content pages:", df["content_id"].nunique())
print("One row = one content page:", len(df) == df["content_id"].nunique())

print("\nTime-window fields available:")
print("- impressions_90d")
print("- clicks_90d")
print("- impressions_last_30d")
print("- impressions_prev_30d")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Working directory: /content/ML_Internship
Dataset exists: True
Rows: 30000
Unique content pages: 30000
One row = one content page: True

Time-window fields available:
- impressions_90d
- clicks_90d
- impressions_last_30d
- impressions_prev_30d


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Fields: feature / label / context / excluded

Features: impressions_prev_30d, content_age_days, days_since_last_update, ctr, avg_position, search_volume, word_count, engagement_rate.

Label: declining_proxy, defined from the observed change between impressions_last_30d and impressions_prev_30d. A page is labeled declining when recent impressions are less than 80% of the previous 30-day impressions.

Context: content_id, content_type, main_intent, competition_level, age_tier, freshness_tier, position_tier.

Excluded: trend_direction and trend_pct are excluded as model features because they are directly related to the outcome being predicted and could cause leakage. client_id is also excluded from model features because the goal is to learn page-level signals rather than memorize client identity.

In [4]:
# This cell is for CODE (numbers, a query, a check).
feature_cols = [
    "impressions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "search_volume",
    "word_count",
    "engagement_rate"
]

context_cols = [
    "content_id",
    "content_type",
    "main_intent",
    "competition_level",
    "age_tier",
    "freshness_tier",
    "position_tier"
]

excluded_cols = [
    "trend_direction",
    "trend_pct",
    "client_id"
]

print("Features:", len(feature_cols))
print("Context fields:", len(context_cols))
print("Excluded fields:", len(excluded_cols))

print("\nAll selected fields exist:",
      all(c in df.columns for c in feature_cols + context_cols + excluded_cols))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Features: 8
Context fields: 7
Excluded fields: 3

All selected fields exist: True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verify it with queries

I will verify the page-level grain, row and unique-page counts, missing values in planned features, and the availability of the 90-day and 30-day performance fields. I will also calculate the declining proxy from the observed impression windows and check its rate.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Grain and counts
print("Rows:", len(df))
print("Unique content pages:", df["content_id"].nunique())

# Missing values in planned features
print("\nMissing values in planned features:")
print(df[feature_cols].isna().sum())

# Verify required time-window fields
window_cols = [
    "impressions_90d",
    "clicks_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d"
]

print("\nTime-window columns available:",
      all(c in df.columns for c in window_cols))

# Declining proxy
df["declining_proxy"] = (
    df["impressions_last_30d"] < 0.8 * df["impressions_prev_30d"]
).astype(int)

print("\nDeclining pages:", df["declining_proxy"].sum())
print("Declining rate:", round(df["declining_proxy"].mean(), 3))

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Rows: 30000
Unique content pages: 30000

Missing values in planned features:
impressions_prev_30d         0
content_age_days             0
days_since_last_update       0
ctr                          0
avg_position                 0
search_volume             2468
word_count                7699
engagement_rate              0
dtype: int64

Time-window columns available: True

Declining pages: 16262
Declining rate: 0.542


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

The data has an unbalanced history, so not every client or page necessarily has the same amount of historical data. Some early rows may contain GSC information without the same depth of other data sources. The 90-day and 30-day windows also overlap in time, so they should not be treated as completely independent observations.

The data can show measured associations and directional patterns in search performance, but it cannot prove why a page declined or prove causality. It also cannot predict or explain Google's ranking algorithm. The output should therefore be treated as decision-support for prioritizing pages for human review, not as an automatic decision.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Check the data limits described above

print("Total pages:", len(df))
print("Unique clients:", df["client_id"].nunique())

# Check whether pages have missing values in selected fields
print("\nMissing values in key fields:")
print(df[[
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "ctr",
    "avg_position"
]].isna().sum())

# Check the overlap of the 30-day windows
print("\nPerformance windows:")
print("Recent window: previous 30 days")
print("Comparison window: preceding 30 days")
print("Combined history represented: approximately 60 days within the 90-day performance fields")

# Confirm that the dataset contains both declining and non-declining pages
print("\nDeclining pages:", df["declining_proxy"].sum())
print("Non-declining pages:", (df["declining_proxy"] == 0).sum())
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Total pages: 30000
Unique clients: 32

Missing values in key fields:
impressions_90d         0
impressions_last_30d    0
impressions_prev_30d    0
ctr                     0
avg_position            0
dtype: int64

Performance windows:
Recent window: previous 30 days
Comparison window: preceding 30 days
Combined history represented: approximately 60 days within the 90-day performance fields

Declining pages: 16262
Non-declining pages: 13738


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.